# Campus Voice — 音声解析と動画生成
Google Colabで実行してください。ランタイム → ランタイムのタイプを変更 → GPUを推奨。
音声はGoogleの実行環境にアップロードされます。保存・再公開の許可を確認してください。
無料枠の計算資源や実行時間は保証されません。初回はモデルをダウンロードします。

初版の候補は発話密度と無音境界に基づく目安です。話題の意味解析・自動話者分離・笑い声検出は未実装です。話者名は編集画面で手入力できます。


In [ ]:
%pip install -q faster-whisper==1.2.1 av==16.0.1
!apt-get update -qq
!apt-get install -y -qq ffmpeg fonts-noto-cjk sentence-transformers==5.1.2


In [ ]:
# Cloudflare公開前でも使える、単体の編集画面です。
from pathlib import Path
from google.colab import files
Path("campus-voice-editor.html").write_text('<!doctype html><html lang="ja"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Campus Voice | 切り抜き編集</title><style>:root{font-family:system-ui,sans-serif;color:#22342f;background:#f5f4ef}body{margin:0}header{background:#173e35;color:white;padding:24px 6%;display:flex;gap:24px;align-items:center}header span{font-weight:800;letter-spacing:3px}main{max-width:920px;margin:48px auto;padding:0 20px}h1{font-size:36px}h2{font-size:20px}section{background:white;border:1px solid #dedfd6;border-radius:16px;padding:28px;margin:24px 0}label{display:block;margin:16px 0;font-size:14px}input,select,textarea{display:block;box-sizing:border-box;width:100%;padding:10px;border:1px solid #bac8bf;border-radius:6px;margin-top:8px;font:inherit}button{background:#23604e;color:white;border:0;border-radius:8px;padding:12px 18px;margin:6px 8px 6px 0;cursor:pointer}button:hover{background:#173e35}audio{width:100%;margin-top:12px}.row{display:flex;gap:20px}.row label{flex:1}.candidate{border-top:1px solid #ddd;padding:14px 0}.segment{border-top:1px solid #eee;padding:10px 0}.segment input{max-width:240px}p,small{line-height:1.7;color:#62746b}header small{color:#c6d9ce}\n\nmain{max-width:1200px}#editor{display:grid;grid-template-columns:minmax(260px,1fr) minmax(320px,1.25fr);gap:24px;align-items:start}#editor[hidden]{display:none}#editor>section{margin:0}#editor>section:last-child{grid-column:1/-1}.candidate{padding:20px;background:#f5f8f5;border:1px solid #d9e4dc;border-radius:12px;margin:12px 0}.candidate h3{margin:0 0 8px;font-size:17px}.candidate p{font-size:13px}#candidates{max-height:75vh;overflow:auto}textarea{min-height:90px;resize:vertical}.secondary{background:#e6eee9;color:#173e35}.secondary:hover{background:#cfdfd5}#duration{font-weight:700;color:#173e35}#saveState{font-weight:700}button:focus-visible,input:focus-visible,select:focus-visible,textarea:focus-visible{outline:3px solid #bd791c;outline-offset:3px}@media(max-width:760px){#editor{display:block}#editor>section{margin:20px 0}.row{flex-wrap:wrap}.row label{min-width:120px}header{flex-wrap:wrap}section{padding:18px}#candidates{max-height:50vh}}\n</style><body><header><span>CAMPUS VOICE</span><small>ラジオのひと場面を、動画に。</small></header><main><h1>切り抜きをつくる</h1><p>Colabで解析 → ここで選んで編集 → Colabで動画を書き出し</p><section><h2>1. 素材を読み込む</h2><label>解析結果・保存した作業（JSON）<input id="project" type="file" accept=".json"></label><label>試聴用の元音声（サーバーへ送信しません）<input id="audioFile" type="file" accept="audio/*"></label><audio id="audio" controls></audio><p id="analysisInfo"></p><p id="status" role="status">解析結果を読み込んでください。</p></section><div id="editor" hidden><section><h2>2. 候補を選ぶ</h2><p>新しいColabでは話題の区切り・発話密度・テンポ・反応語で候補を提案します。反応語は笑い声の検出ではありません。自動話者分離は未実装です。自動提案は確認用の目安です。前後を聴いて内容を判断してください。</p><div class="row"><label>候補・字幕を検索<input id="search" type="search" placeholder="気になる言葉を入力"></label><label>並べ替え<select id="sort"><option value="score">おすすめ順</option><option value="time">時刻順</option></select></label></div><p id="candidateCount"></p><div id="candidates"></div></section><section><h2>3. 内容を整える</h2><label>タイトル<input id="title"></label><div class="row"><label>開始（秒）<input id="start" type="number" min="0" step="0.1"></label><label>終了（秒）<input id="end" type="number" min="0" step="0.1"></label></div><p id="duration" role="status"></p><button id="useStart" class="secondary">再生位置を開始にする</button><button id="useEnd" class="secondary">再生位置を終了にする</button><br><button id="listen">選択範囲を試聴</button><button id="context">開始の10秒前から聴く</button><p>字幕は直接修正できます。解析時の話者名がある場合は、各欄で名前も変更できます。</p><div id="segments"></div></section><section><h2>4. 書き出し設定</h2><label>動画サイズ<select id="format"><option value="vertical">縦型 9:16（Shorts / Reels）</option><option value="square">正方形 1:1</option><option value="landscape">横型 16:9</option></select></label><label>BGM音量（元音声に対する倍率）<input id="volume" type="number" min="0" max="1" step="0.01" value="0.12"></label><p>BGMファイルは動画生成時にColabで選びます。利用できる権利のある音源をご用意ください。</p><p id="saveState" role="status">読み込み済み</p><button id="saveSession">作業全体を保存（再編集用）</button><button id="download">Colab用の編集JSONを保存</button><p>保存したJSONと元音声をColabに読み込み、MP4を生成します。</p></section></div><button id="clear">読み込みデータを消去</button></main><script type="module">function validateProject(p) {\n  if (typeof p?.audio_name !== \'string\' || !p.audio_name || !p.segments?.length) throw Error(\'元音声名と字幕が必要です\');\n  if (p?.version !== 1 || !Array.isArray(p.segments) || !Array.isArray(p.candidates)) throw Error(\'対応する解析結果ではありません\');\n  for (const s of p.segments) if (!Number.isFinite(s.start) || !Number.isFinite(s.end) || s.start < 0 || s.end <= s.start || typeof s.text !== \'string\') throw Error(\'字幕の時刻・内容が不正です\');\n  for (const c of p.candidates) if (!Number.isFinite(c.start) || !Number.isFinite(c.end) || c.start < 0 || c.end <= c.start) throw Error(\'候補の時刻が不正です\');\n  return p;\n}\nfunction exportEdit(project, start, end, segments, bgmVolume, format) {\n  if (![\'vertical\',\'square\',\'landscape\'].includes(format)) throw Error(\'動画サイズが不正です\');\n  if (!Number.isFinite(bgmVolume) || bgmVolume < 0 || bgmVolume > 1) throw Error(\'BGM音量は0〜1にしてください\');\n  if (!Number.isFinite(start) || !Number.isFinite(end) || start < 0 || end <= start) throw Error(\'終了時刻は開始時刻より後にしてください\');\n  if (!segments.some(s => s.end > start && s.start < end)) throw Error(\'字幕のある範囲を選んでください\');\n  return {version:1, audio_name:project.audio_name, start,end,segments:segments.filter(s=>s.end>start&&s.start<end),bgm_volume:bgmVolume,format};\n}\n\n\nconst $=id=>document.getElementById(id);\nlet project,url,stopAt,dirty=false;\nfunction status(message){$(\'status\').textContent=message}\nfunction changed(){dirty=true;$(\'saveState\').textContent=\'未保存の変更があります\'}\nfunction range(start,end){$(\'start\').value=start;$(\'end\').value=end;renderSegments();updateDuration()}\nfunction renderSegments(){\n  const root=$(\'segments\');root.replaceChildren();\n  for(const s of project.segments){\n    if(s.end<=+$(\'start\').value||s.start>=+$(\'end\').value)continue;\n    const box=document.createElement(\'div\');box.className=\'segment\';\n    const time=document.createElement(\'small\');time.textContent=`${s.start.toFixed(1)}–${s.end.toFixed(1)}秒`;\n    const speaker=document.createElement(\'input\');speaker.placeholder=\'話者名（任意）\';speaker.setAttribute(\'aria-label\',\'話者名\');speaker.value=s.speaker||\'\';\n    speaker.oninput=()=>{s.speaker=speaker.value;changed()};\n    const text=document.createElement(\'textarea\');text.setAttribute(\'aria-label\',`${s.start.toFixed(1)}秒の字幕`);text.value=s.text;\n    text.oninput=()=>{s.text=text.value;changed()};box.append(time,speaker,text);root.append(box);\n  }\n}\nfunction loadProject(p,selection){\n  project=validateProject(p);$(\'editor\').hidden=false;\n  renderCandidates();\n  const c=selection||project.candidates[0]||{start:project.segments[0].start,end:Math.min(project.segments.at(-1).end,project.segments[0].start+60)};\n  range(c.start,c.end);$(\'title\').value=c.title||\'\';$(\'volume\').value=c.bgm_volume??0.12;$(\'format\').value=c.format||\'vertical\';\n  dirty=false;$(\'saveState\').textContent=\'読み込み済み\';\n  $(\'analysisInfo\').textContent=project.analysis?.test_seconds ? `冒頭${project.analysis.test_seconds}秒の解析です。全編ではありません。` : (project.analysis?.test_seconds===0 ? \'全編を解析したデータです。\' : \'解析範囲は不明です。\');\n}\n$(\'project\').onchange=async e=>{\n  const file=e.target.files[0];if(!file)return;\n  if(dirty&&!confirm(\'未保存の編集があります。新しいデータを読み込みますか？\'))return;\n  try{\n    const data=JSON.parse(await file.text());\n    if(data.kind===\'campus-voice-session\'){\n      validateProject(data.project);\n      exportEdit(data.project,data.selection.start,data.selection.end,data.project.segments,data.selection.bgm_volume,data.selection.format);\n      loadProject(data.project,data.selection);\n    }else loadProject(data);\n    status(\'読み込みました。試聴するには元音声を選択してください。\');\n  }catch(e){status(e.message)}\n};\n$(\'audioFile\').onchange=e=>{\n  const file=e.target.files[0];if(!file)return;\n  if(project&&file.name!==project.audio_name){status(`元音声は「${project.audio_name}」を選んでください。`);e.target.value=\'\';return}\n  $(\'audio\').pause();stopAt=null;if(url)URL.revokeObjectURL(url);\n  url=URL.createObjectURL(file);$(\'audio\').src=url;status(\'音声を読み込みました。候補の前後を確認してください。\');\n};\nfor(const id of [\'start\',\'end\'])$(id).onchange=()=>{renderSegments();updateDuration();changed()};\nfor(const id of [\'title\',\'volume\',\'format\'])$(id).oninput=changed;\nasync function listen(context=false){\n  try{\n    const start=+$(\'start\').value,end=+$(\'end\').value;\n    exportEdit(project,start,end,project.segments,+$(\'volume\').value,$(\'format\').value);\n    if(!$(\'audio\').src||!Number.isFinite($(\'audio\').duration))throw Error(\'元音声を読み込み、準備ができるまでお待ちください\');\n    if(end>$(\'audio\').duration+0.1)throw Error(\'選択範囲が音声の長さを超えています\');\n    stopAt=Math.min($(\'audio\').duration,end+(context?10:0));$(\'audio\').currentTime=Math.max(0,start-(context?10:0));\n    await $(\'audio\').play();\n  }catch(e){status(e.message)}\n}\n$(\'listen\').onclick=()=>listen();$(\'context\').onclick=()=>listen(true);\n$(\'audio\').ontimeupdate=()=>{if(stopAt&&$(\'audio\').currentTime>=stopAt){$(\'audio\').pause();stopAt=null}};\nfunction selection(){const edit=exportEdit(project,+$(\'start\').value,+$(\'end\').value,project.segments,+$(\'volume\').value,$(\'format\').value);edit.title=$(\'title\').value;return edit}\nfunction download(data,name){const u=URL.createObjectURL(new Blob([JSON.stringify(data,null,2)],{type:\'application/json\'}));const a=document.createElement(\'a\');a.href=u;a.download=name;a.click();setTimeout(()=>URL.revokeObjectURL(u),1000)}\n$(\'download\').onclick=()=>{try{download(selection(),\'clip-edit.json\');status(\'Colab用の編集JSONを保存しました。再編集用には「作業全体を保存」も使ってください。\')}catch(e){status(e.message)}};\n$(\'saveSession\').onclick=()=>{try{download({kind:\'campus-voice-session\',version:1,project,selection:selection()},\'campus-voice-session.json\');dirty=false;$(\'saveState\').textContent=\'作業を保存しました\';status(\'作業全体を保存しました。次回はこのJSONと元音声を読み込むと再開できます。\')}catch(e){status(e.message)}};\n$(\'clear\').onclick=()=>{\n  if(dirty&&!confirm(\'未保存の編集を破棄して消去しますか？\'))return;\n  project=null;dirty=false;stopAt=null;$(\'editor\').hidden=true;$(\'audio\').pause();$(\'audio\').removeAttribute(\'src\');$(\'audio\').load();\n  if(url)URL.revokeObjectURL(url);url=null;$(\'project\').value=\'\';$(\'audioFile\').value=\'\';$(\'candidates\').replaceChildren();$(\'segments\').replaceChildren();\n  status(\'読み込んだデータを消去しました。端末に保存済みのファイルは別途削除してください。\');\n};\nwindow.addEventListener(\'beforeunload\',e=>{if(dirty){e.preventDefault();e.returnValue=\'\'}});\n\nfunction updateDuration(){\n  const duration=+$(\'end\').value-+$(\'start\').value;\n  $(\'duration\').textContent=Number.isFinite(duration)&&duration>0 ? `${duration.toFixed(1)}秒${duration<30||duration>60?\' · おすすめは30〜60秒です\':\'\'}` : \'開始・終了を確認してください\';\n}\nfunction renderCandidates(){\n  const root=$(\'candidates\');root.replaceChildren();\n  if(!project)return;\n  const query=$(\'search\').value.trim().toLowerCase();\n  const candidates=project.candidates.filter(c=>`${c.title||\'\'} ${c.reason||\'\'} ${project.segments.filter(s=>s.end>c.start&&s.start<c.end).map(s=>s.text).join(\' \')}`.toLowerCase().includes(query));\n  candidates.sort($(\'sort\').value===\'time\' ? (a,b)=>a.start-b.start : (a,b)=>(b.score||0)-(a.score||0));\n  $(\'candidateCount\').textContent=`${candidates.length}件の候補`;\n  if(!candidates.length){const empty=document.createElement(\'p\');empty.textContent=\'候補がありません。検索を変えるか、開始・終了を手動で指定してください。\';root.append(empty)}\n  for(const [i,c] of candidates.entries()){\n    const card=document.createElement(\'article\');card.className=\'candidate\';\n    const heading=document.createElement(\'h3\');heading.textContent=`${i+1}. ${c.title||\'候補\'}`;\n    const detail=document.createElement(\'small\');detail.textContent=`${clock(c.start)}〜${clock(c.end)} · ${(c.end-c.start).toFixed(1)}秒`;\n    const reason=document.createElement(\'p\');reason.textContent=c.reason||\'前後を聴いて確認してください\';\n    const select=()=>{range(c.start,c.end);$(\'title\').value=c.title||\'\';changed()};\n    const edit=document.createElement(\'button\');edit.textContent=\'この範囲を編集\';edit.onclick=()=>{select();$(\'title\').focus()};\n    const preview=document.createElement(\'button\');preview.className=\'secondary\';preview.textContent=\'前後を試聴\';preview.onclick=()=>{select();listen(true)};\n    card.append(heading,detail,reason,edit,preview);root.append(card);\n  }\n}\nfunction clock(value){return `${Math.floor(value/60)}:${Math.floor(value%60).toString().padStart(2,\'0\')}`}\n$(\'search\').oninput=renderCandidates;$(\'sort\').onchange=renderCandidates;\n$(\'useStart\').onclick=()=>{if(Number.isFinite($(\'audio\').duration)){range(Number($(\'audio\').currentTime.toFixed(1)),+$(\'end\').value);changed()}else status(\'元音声を先に選択してください\')};\n$(\'useEnd\').onclick=()=>{if(Number.isFinite($(\'audio\').duration)){range(+$(\'start\').value,Number($(\'audio\').currentTime.toFixed(1)));changed()}else status(\'元音声を先に選択してください\')};\n</script></body></html>\n', encoding="utf-8")
files.download("campus-voice-editor.html")


## 1. 元音声をアップロード（解析時・動画生成時の両方で必要）

In [ ]:
from google.colab import files
from pathlib import Path
import json, subprocess, math
uploaded = files.upload()
audio_files = [Path(name) for name in uploaded if Path(name).suffix.lower() in {'.mp3','.wav','.m4a','.ogg','.flac'}]
assert len(audio_files) == 1, '音声を1本選択してください'
audio_path = audio_files[0]
del uploaded


## 2. 文字起こしと候補抽出（解析済みならスキップ可）
日本語モデルで解析します。GPUが使えない場合はCPUで実行するため時間がかかります。
初回は冒頭180秒のみ。確認後は `TEST_SECONDS = 0` に変更すると全編を解析します。処理中はチェックポイントを保存しますが、ランタイムが削除されると失われるため必要ならダウンロードしてください。


In [ ]:
from faster_whisper import WhisperModel
import shutil
has_gpu = bool(shutil.which('nvidia-smi')) and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
model = WhisperModel('small', device='cuda' if has_gpu else 'cpu', compute_type='float16' if has_gpu else 'int8')
# 初回は冒頭3分で動作確認。全編を解析するときは0にして再実行。
TEST_SECONDS = 180
source_path = audio_path
if TEST_SECONDS:
    source_path = Path('trial-audio.wav')
    subprocess.run(['ffmpeg','-y','-v','error','-i',str(audio_path),'-t',str(TEST_SECONDS),'-ar','16000','-ac','1',str(source_path)],check=True)
result, info = model.transcribe(str(source_path), language='ja', vad_filter=True, beam_size=5)
segments = []
try:
    for item in result:
        if not item.text.strip(): continue
        segments.append({'start':item.start,'end':item.end,'text':item.text.strip(),'speaker':''})
        Path('transcript-checkpoint.json').write_text(json.dumps({'audio_name':audio_path.name,'segments':segments},ensure_ascii=False),encoding='utf-8')
        print(f'文字起こし済み: {item.end:.0f}秒', end='\r')
except Exception:
    if segments:
        print('途中結果を保存します。これは完成した解析JSONではありません。')
        files.download('transcript-checkpoint.json')
    raise
assert segments, '発話を検出できませんでした'


In [ ]:
"""Candidate ranking; optional embeddings are computed in Colab, never the editor."""
import math
import re


def suggest_candidates(segments, embeddings=None, limit=10):
    """Rank 30–60s windows with topic, tempo and transcript reaction cues.

    Embeddings must align with segments. Reaction words are not laughter detection.
    Short recordings return fewer candidates rather than duplicating intervals.
    """
    if embeddings is not None and len(embeddings) != len(segments):
        raise ValueError('字幕と埋め込みの件数が一致しません')
    if not segments:
        return []
    boundaries = set()
    if embeddings is not None:
        for i in range(1, len(segments)):
            a, b = embeddings[i - 1], embeddings[i]
            norm = math.sqrt(sum(x*x for x in a) * sum(x*x for x in b))
            similarity = sum(x*y for x, y in zip(a, b)) / norm if norm else 1
            if similarity < .45:
                boundaries.add(i)
    windows = []
    for i, first in enumerate(segments):
        # A long individual subtitle cannot make a 30–60s candidate safely.
        selected = []
        for j in range(i, len(segments)):
            if segments[j]['end'] - first['start'] > 60:
                break
            selected.append(segments[j])
            duration = segments[j]['end'] - first['start']
            if duration < 30:
                continue
            gap = segments[j+1]['start']-segments[j]['end'] if j+1 < len(segments) else 2
            clean_end = gap >= .8 or j+1 in boundaries or j == len(segments)-1
            text = ''.join(s['text'] for s in selected)
            speech = sum(s['end']-s['start'] for s in selected)
            density = min(1, speech / duration)
            tempo = min(1, len(selected) / duration * 5)
            reactions = len(re.findall(r'すごい|面白|おもしろ|まさか|なるほど|[!！]|[（(]笑[)）]', text))
            topic = i in boundaries or i == 0
            score = .4*density + .25*tempo + .15*min(1,reactions/3) + .1*topic + .1*clean_end
            reasons = [f'発話密度 {density:.0%}', f'発話区間 {len(selected)}件']
            if topic and embeddings is not None:
                reasons.append('話題の区切りから開始')
            if reactions:
                reasons.append(f'反応語 {reactions}件（文字情報）')
            if clean_end:
                reasons.append('区切りで終了')
            windows.append(dict(start=first['start'], end=segments[j]['end'],
                                title=first['text'][:40], reason=' / '.join(reasons),
                                score=round(score,4), density=round(density,3),
                                reactions=reactions, topic_start=bool(topic and embeddings is not None)))
    chosen = []
    for candidate in sorted(windows, key=lambda c: c['score'], reverse=True):
        # Prevent near-duplicates while allowing a short shared lead-in.
        if any(max(0,min(candidate['end'], c['end'])-max(candidate['start'],c['start'])) /
               min(candidate['end']-candidate['start'],c['end']-c['start']) > .25 for c in chosen):
            continue
        chosen.append(candidate)
        if len(chosen) >= limit:
            break
    return chosen

# 候補だけ再計算する場合は、このセルから実行します。
USE_TOPIC_ANALYSIS = True
CANDIDATE_COUNT = 10  # 5〜10。短い音声では候補が少なくなることがあります。
topic_embeddings = None
if USE_TOPIC_ANALYSIS:
    try:
        from sentence_transformers import SentenceTransformer
        topic_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device='cpu')
        topic_embeddings = topic_model.encode([s['text'] for s in segments], normalize_embeddings=True).tolist()
        del topic_model
    except Exception as error:
        print('話題モデルを利用できないため、発話密度・テンポ・反応語で候補を作ります:', type(error).__name__)
if not 5 <= CANDIDATE_COUNT <= 10:
    raise ValueError('候補数は5〜10にしてください')
candidates = suggest_candidates(segments, topic_embeddings, CANDIDATE_COUNT)
if not candidates:
    print('30〜60秒の候補がありません。編集画面で範囲を手動指定してください。')
project={'version':1,'audio_name':audio_path.name,'segments':segments,'candidates':candidates,'analysis':{'test_seconds':TEST_SECONDS,'speaker_diarization':False,'laughter_detection':False,'topic_analysis':bool(topic_embeddings is not None),'candidate_method':'topic-tempo-reactions-v2'}}
Path('analysis.json').write_text(json.dumps(project,ensure_ascii=False,indent=2))
files.download('analysis.json')
print(f'{len(segments)}字幕、{len(candidates)}候補を保存しました。編集画面へ読み込んでください。')


## 3. 編集JSONと任意のBGMをアップロード
編集画面で保存したclip-edit.jsonを選択。BGMを使う場合は同時に音声ファイルを選んでください。

In [ ]:
uploaded = files.upload()
json_names=[n for n in uploaded if n.endswith('.json')]
assert len(json_names)==1, '編集JSONを1つ選択してください'
edit=json.loads(uploaded[json_names[0]])
assert edit.get('version')==1 and 0<=edit['start']<edit['end']
assert edit['audio_name']==audio_path.name, '解析時と同じ元音声を選択してください'
bgm_names=[n for n in uploaded if Path(n).suffix.lower() in {'.mp3','.wav','.m4a','.ogg','.flac'}]
assert len(bgm_names)<=1, 'BGMは1本までです'
bgm=bgm_names[0] if bgm_names else None
del uploaded


## 4. 字幕付きMP4を生成
背景は単色の仮デザインです。BGMには音量調整・冒頭と末尾のフェードを適用します。

In [ ]:
start,end=float(edit['start']),float(edit['end']); duration=end-start
assert math.isfinite(duration) and 0<duration<=600
size={'vertical':(1080,1920),'square':(1080,1080),'landscape':(1920,1080)}[edit['format']]
def stamp(t):
    cs=max(0,round(t*100)); h,cs=divmod(cs,360000); m,cs=divmod(cs,6000); s,cs=divmod(cs,100)
    return f'{h}:{m:02}:{s:02}.{cs:02}'
def safe_text(t):
    return str(t).replace('\\','').replace('{','').replace('}','').replace('\n',r'\N').replace('\r','')
w,h=size
header=f"""[Script Info]
ScriptType: v4.00+
PlayResX: {w}
PlayResY: {h}
WrapStyle: 0
[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,Noto Sans CJK JP,56,&H00FFFFFF,&H00FFFFFF,&H00203930,&H00203930,0,0,0,0,100,100,0,0,1,3,0,5,80,80,60,1
[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
lines=[]
for s in edit['segments']:
    a=max(start,float(s['start'])); b=min(end,float(s['end']))
    if b<=a: continue
    text=(str(s.get('speaker',''))+'：' if s.get('speaker') else '')+str(s['text'])
    lines.append(f'Dialogue: 0,{stamp(a-start)},{stamp(b-start)},Default,,0,0,0,,{safe_text(text)}')
Path('captions.ass').write_text(header+'\n'.join(lines),encoding='utf-8')
cmd=['ffmpeg','-y','-f','lavfi','-i',f'color=c=0x173e35:s={w}x{h}:r=30:d={duration}', '-ss',str(start),'-t',str(duration),'-i',str(audio_path)]
if bgm:
    volume=float(edit.get('bgm_volume',0.12)); assert 0<=volume<=1
    cmd+=['-stream_loop','-1','-i',bgm,'-filter_complex',f'[2:a]volume={volume},afade=t=in:d=1,afade=t=out:st={max(0,duration-1)}:d=1[bg];[1:a][bg]amix=inputs=2:duration=first:normalize=0[a]','-map','0:v','-map','[a]']
else: cmd+=['-map','0:v','-map','1:a']
cmd+=['-vf','ass=captions.ass','-t',str(duration),'-c:v','libx264','-preset','fast','-threads','2','-pix_fmt','yuv420p','-c:a','aac','-movflags','+faststart','clip.mp4']
subprocess.run(cmd,check=True)
files.download('clip.mp4')


## データの削除
ダウンロードしたファイルは手元で削除してください。Colab上の元音声などは、ランタイムの「接続を解除して削除」で実行環境ごと削除できます。Driveには自動保存しません。